# Лабораторная работа № 3.1
## Проверка гипотезы о нормальном распределении

**Выполнили:**
- Левченков Артемий Викторович;
- Бикбулатова Динара Фануровна;
- Ладыгин Максим Александрович.

**Группа:** БИВТ-24-АЭ-3

**Цель работы:** по критерию согласия Пирсона при уровне значимости $\alpha=0{,}05$ проверить гипотезу $H_0$ о нормальном распределении возраста и среднего возраста людей, совершивших преступления в Москве в 2021 году.

### Теоретическая часть

$H_0$: распределение исследуемой случайной величины является нормальным. $H_1$: распределение не является нормальным. Для интервального ряда с серединами $x_i^*$ и частотами $n_i$:

$$x_i^*=\frac{x_i+x_{i+1}}{2},\qquad \bar{x}^*=\frac{\sum x_i^*n_i}{n},\qquad \sigma^*=\sqrt{\frac{\sum (x_i^*-\bar{x}^*)^2n_i}{n}}.$$

Границы интервалов нормируются:
$$z_i=\frac{x_i-\bar{x}^*}{\sigma^*},\qquad z_{i+1}=\frac{x_{i+1}-\bar{x}^*}{\sigma^*}.$$

Для крайних интервалов $z_1=-\infty$, $z_s=+\infty$. Вероятности и теоретические частоты:
$$p_i=\Phi(z_{i+1})-\Phi(z_i),\qquad n_i'=np_i.$$

Критерий Пирсона и число степеней свободы:
$$\chi^2_{\text{набл}}=\sum_{i=1}^{s}\frac{(n_i-n_i')^2}{n_i'},\qquad k=s-1-r=s-3,$$

где $s$ — число групп, а $r=2$ — число оцениваемых параметров нормального распределения ($\bar{x}^*$ и $\sigma^*$). Если $\chi^2_{\text{набл}}<\chi^2_{\text{кр}}$, оснований отвергать $H_0$ нет; иначе $H_0$ отвергается.

In [1]:
import pandas as pd
from math import ceil, floor, sqrt
from scipy.stats import chi2, norm

data = pd.read_table("data/moscow.txt", header=None)[0]
N = len(data)

print("Количество наблюдений:", N)
print("Минимальный возраст:", data.min())
print("Максимальный возраст:", data.max())

Количество наблюдений: 32423
Минимальный возраст: 14
Максимальный возраст: 73


### 1. Интервальный ряд возраста

Используется интервальный ряд из 7 групп, построенный в ЛР 1.

In [2]:
k_age = 7
x_min_age = data.min()
x_max_age = data.max()
h_age = ceil((x_max_age - x_min_age) / k_age)
edges_age = [x_min_age + i * h_age for i in range(k_age + 1)]

rows = []
for i in range(k_age):
    left = edges_age[i]
    right = edges_age[i + 1]

    if i < k_age - 1:
        n = ((data >= left) & (data < right)).sum()
    else:
        n = ((data >= left) & (data <= right)).sum()

    rows.append([left, right, n])

age_interval = pd.DataFrame(rows, columns=["x_l", "x_r", "n"])
display(age_interval)

,x_l,x_r,n
0,14,23,4811
1,23,32,9476
2,32,41,7243
3,41,50,7951
4,50,59,1140
5,59,68,1472
6,68,77,330


### 2. Интервальный ряд среднего возраста

Объём повторной выборки:

$$n=\left\lceil\frac{t^2\sigma^2}{\delta^2}\right\rceil.$$

Формируются 36 повторных выборок без фиксации генератора случайных чисел.


In [3]:
def mean_value(x, n):
    return sum(x * n) / sum(n)


def dispersion(x, n, mean):
    return sum((x - mean) ** 2 * n) / sum(n)


discrete = data.value_counts().sort_index().rename_axis("x").reset_index(name="n")
x = discrete["x"]
n = discrete["n"]
mean_age = mean_value(x, n)
D_age = dispersion(x, n, mean_age)

gamma = 0.95
delta = 3
t = 1.96
sample_size = ceil(t ** 2 * D_age / delta ** 2)
number_of_samples = 36

sample_means = []
for i in range(number_of_samples):
    sample = data.sample(n=sample_size, replace=True)
    sample_means.append(sample.mean())

sample_means = pd.Series(sample_means)
x_min_mean = floor(sample_means.min())
x_max_mean = ceil(sample_means.max())
if x_min_mean == x_max_mean:
    x_max_mean += 1

h_mean = 1
k_mean = x_max_mean - x_min_mean
edges_mean = [x_min_mean + i * h_mean for i in range(k_mean + 1)]

rows = []
for i in range(k_mean):
    left = edges_mean[i]
    right = edges_mean[i + 1]

    if i < k_mean - 1:
        n = ((sample_means >= left) & (sample_means < right)).sum()
    else:
        n = ((sample_means >= left) & (sample_means <= right)).sum()

    rows.append([left, right, n])

mean_interval = pd.DataFrame(rows, columns=["x_l", "x_r", "n"])

print(f"Объём каждой выборки: {sample_size}")
print(f"Количество выборок: {number_of_samples}")
print(f"Минимальное выборочное среднее: {sample_means.min():.4f}")
print(f"Максимальное выборочное среднее: {sample_means.max():.4f}")
display(mean_interval)

Объём каждой выборки: 62
Количество выборок: 36
Минимальное выборочное среднее: 32.0968
Максимальное выборочное среднее: 38.1935


,x_l,x_r,n
0,32,33,2
1,33,34,2
2,34,35,13
3,35,36,10
4,36,37,6
5,37,38,2
6,38,39,1


### 3. Проверка гипотезы по критерию Пирсона

Функция применяется к обоим рядам; число групп $s$ определяется по исходному интервальному ряду.

In [4]:
def check_normal_distribution(interval_distribution, alpha, title):
    required_columns = {"x_l", "x_r", "n"}
    if not required_columns.issubset(interval_distribution.columns):
        raise ValueError("Нужны столбцы x_l, x_r и n.")
    if not 0 < alpha < 1:
        raise ValueError("Уровень значимости должен находиться между 0 и 1.")

    table = interval_distribution[["x_l", "x_r", "n"]].copy().reset_index(drop=True)
    s = len(table)
    if s <= 3:
        raise ValueError("Для критерия Пирсона нужно больше трёх групп.")
    if (table["n"] < 0).any() or table["n"].sum() == 0:
        raise ValueError("Частоты должны быть неотрицательными, а их сумма — положительной.")

    widths = table["x_r"] - table["x_l"]
    if (widths <= 0).any() or not (widths == widths.iloc[0]).all():
        raise ValueError("Интервалы должны иметь одинаковую положительную длину.")

    total = table["n"].sum()
    table["i"] = range(1, s + 1)
    table["x_middle"] = (table["x_l"] + table["x_r"]) / 2

    mean = mean_value(table["x_middle"], table["n"])
    sigma = sqrt(dispersion(table["x_middle"], table["n"], mean))
    if sigma == 0:
        raise ValueError("СКО равно нулю: нормирование выполнить нельзя.")

    table["x_l - mean"] = table["x_l"] - mean
    table["x_r - mean"] = table["x_r"] - mean
    table["z_l"] = table["x_l - mean"] / sigma
    table["z_r"] = table["x_r - mean"] / sigma
    table.loc[0, "z_l"] = float("-inf")
    table.loc[s - 1, "z_r"] = float("inf")

    table["Phi(z_l)"] = norm.cdf(table["z_l"]) - 0.5
    table["Phi(z_r)"] = norm.cdf(table["z_r"]) - 0.5
    table["p"] = table["Phi(z_r)"] - table["Phi(z_l)"]
    table["n_theory"] = total * table["p"]
    table["chi_part"] = (table["n"] - table["n_theory"]) ** 2 / table["n_theory"]

    chi_observed = table["chi_part"].sum()
    degrees_of_freedom = s - 3
    chi_critical = chi2.ppf(1 - alpha, degrees_of_freedom)
    hypothesis_not_rejected = chi_observed < chi_critical

    print(title)
    print(f"Объём ряда n = {total}, число групп s = {s}")
    print(f"Средняя x̄* = {mean:.4f}, СКО σ* = {sigma:.4f}")
    display(table[["i", "x_l", "x_r", "x_middle", "x_l - mean", "x_r - mean",
                   "z_l", "z_r", "Phi(z_l)", "Phi(z_r)", "p", "n",
                   "n_theory", "chi_part"]].round(4))
    print(f"χ²набл = {chi_observed:.4f}")
    print(f"Число степеней свободы: k = s - 3 = {degrees_of_freedom}")
    print(f"χ²кр = {chi_critical:.4f} при α = {alpha}")

    if hypothesis_not_rejected:
        print("χ²набл < χ²кр: оснований отвергать H0 нет. Данные согласуются с нормальным распределением.")
    else:
        print("χ²набл ≥ χ²кр: H0 отвергается. Данные не согласуются с нормальным распределением.")

    small_expected = table.loc[table["n_theory"] < 5, "i"].tolist()
    if small_expected:
        print(f"В группах {small_expected} теоретическая частота меньше 5; результат следует интерпретировать с осторожностью.")

    return {
        "table": table,
        "chi_observed": chi_observed,
        "chi_critical": chi_critical,
        "degrees_of_freedom": degrees_of_freedom,
        "hypothesis_not_rejected": hypothesis_not_rejected,
    }

### 4. Проверка двух рядов

In [5]:
age_result = check_normal_distribution(
    age_interval,
    alpha=0.05,
    title="Проверка распределения возраста",
)

print("\n" + "=" * 100 + "\n")

mean_result = check_normal_distribution(
    mean_interval,
    alpha=0.05,
    title="Проверка распределения среднего возраста",
)

Проверка распределения возраста
Объём ряда n = 32423, число групп s = 7
Средняя x̄* = 35.6309, СКО σ* = 12.1875


,i,x_l,x_r,x_middle,x_l - mean,x_r - mean,z_l,z_r,Phi(z_l),Phi(z_r),p,n,n_theory,chi_part
0,1,14,23,18.5,-21.6309,-12.6309,-inf,-1.0364,-0.5000,-0.3500,0.1500,4811,4863.8722,0.5747
1,2,23,32,27.5,-12.6309,-3.6309,-1.0364,-0.2979,-0.3500,-0.1171,0.2329,9476,7550.3325,491.1301
2,3,32,41,36.5,-3.6309,5.3691,-0.2979,0.4405,-0.1171,0.1702,0.2873,7243,9316.5699,461.5102
3,4,41,50,45.5,5.3691,14.3691,0.4405,1.1790,0.1702,0.3808,0.2106,7951,6827.4299,184.9026
4,5,50,59,54.5,14.3691,23.3691,1.1790,1.9175,0.3808,0.4724,0.0916,1140,2970.2496,1127.7886
5,6,59,68,63.5,23.3691,32.3691,1.9175,2.6559,0.4724,0.4960,0.0236,1472,766.3244,649.8268
6,7,68,77,72.5,32.3691,41.3691,2.6559,inf,0.4960,0.5000,0.0040,330,128.2216,317.5326


χ²набл = 3233.2657
Число степеней свободы: k = s - 3 = 4
χ²кр = 9.4877 при α = 0.05
χ²набл ≥ χ²кр: H0 отвергается. Данные не согласуются с нормальным распределением.


Проверка распределения среднего возраста
Объём ряда n = 36, число групп s = 7
Средняя x̄* = 35.2222, СКО σ* = 1.2826


,i,x_l,x_r,x_middle,x_l - mean,x_r - mean,z_l,z_r,Phi(z_l),Phi(z_r),p,n,n_theory,chi_part
0,1,32,33,32.5,-3.2222,-2.2222,-inf,-1.7326,-0.5000,-0.4584,0.0416,2,1.4970,0.1690
1,2,33,34,33.5,-2.2222,-1.2222,-1.7326,-0.9529,-0.4584,-0.3297,0.1287,2,4.6343,1.4974
2,3,34,35,34.5,-1.2222,-0.2222,-0.9529,-0.1733,-0.3297,-0.0688,0.2609,13,9.3928,1.3853
3,4,35,36,35.5,-0.2222,0.7778,-0.1733,0.6064,-0.0688,0.2279,0.2967,10,10.6795,0.0432
4,5,36,37,36.5,0.7778,1.7778,0.6064,1.3861,0.2279,0.4171,0.1893,6,6.8134,0.0971
5,6,37,38,37.5,1.7778,2.7778,1.3861,2.1657,0.4171,0.4848,0.0677,2,2.4371,0.0784
6,7,38,39,38.5,2.7778,3.7778,2.1657,inf,0.4848,0.5000,0.0152,1,0.5460,0.3776


χ²набл = 3.6481
Число степеней свободы: k = s - 3 = 4
χ²кр = 9.4877 при α = 0.05
χ²набл < χ²кр: оснований отвергать H0 нет. Данные согласуются с нормальным распределением.
В группах [1, 2, 6, 7] теоретическая частота меньше 5; результат следует интерпретировать с осторожностью.
